<a href="https://colab.research.google.com/github/hilalozkaraman/leukemia-ml-llm-audit/blob/main/leukemia_ml_llm_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import os
import re
import io
import json
import gzip
import zipfile
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import false_discovery_control  # scipy >= 1.11 (Benjamini-Hochberg)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score


In [24]:
# 1. SABİT RANDOM SEED VE GÖRSEL AYARLAR (IEEE Kalite Kıstası: >= 300 DPI veya vektör)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 10
sns.set_theme(style="whitegrid", palette="deep")

OUTPUT_DIR = "./eda_outputs"
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

RESULTS = {"golub": {}, "gse13159": {}, "cross_dataset": {}}  # rapordaki tüm sayılar buraya yazılır


In [25]:
# ==============================================================================
# ADIM 0: GOOGLE DRIVE BAĞLANTISI VE DOSYA YOLLARI
# ==============================================================================
from google.colab import drive
drive.mount('/content/drive')

GOLUB_ZIP_PATH = "/content/drive/MyDrive/archive.zip"  # Kaggle Golub zip dosyası
GSE13159_GZ_PATH = "/content/drive/MyDrive/GSE13159_series_matrix.txt.gz"  # GEO dosyası
GPL570_PATH = "/content/drive/MyDrive/GPL570.annot.gz"  # (opsiyonel) ortak gen analizi için GEO anotasyonu
GOLUB_MAP_CSV = None  # (opsiyonel) 'probe,symbol' sütunlu CSV; Golub prob->gen sembolü eşlemesini ezer
EXTRACT_DIR = "./data_golub"

FLOOR, CEIL = 20.0, 16000.0  # Rehber 6.1: örnek alt-üst eşik
PAL_CLASS = {"ALL": "#2b83ba", "AML": "#d7191c"}


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
# ==============================================================================
# YARDIMCI FONKSİYONLAR
# ==============================================================================
def save_fig(name):
    """Şekli hem PNG (300 dpi) hem vektör PDF olarak kaydeder."""
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{name}.png"))
    plt.savefig(os.path.join(OUTPUT_DIR, f"{name}.pdf"))
    plt.close()


def save_clustermap(g, name):
    g.savefig(os.path.join(OUTPUT_DIR, f"{name}.png"))
    g.savefig(os.path.join(OUTPUT_DIR, f"{name}.pdf"))
    plt.close()


def clean_for_eda(X_raw, train_idx=None, is_log2=False, floor=FLOOR, ceil=CEIL):
    """BETİMSEL analiz için: INF->NaN, (eğitim) ortalaması ile doldur, eşik kırp, log2.
    Model hattında bu adımlar CV katı içinde, yalnızca eğitim verisiyle yeniden yapılmalıdır."""
    X = X_raw.replace([np.inf, -np.inf], np.nan)
    means = X.loc[train_idx].mean(axis=0) if train_idx is not None else X.mean(axis=0)
    X = X.fillna(means).fillna(0.0)  # ikinci fillna: tüm sütunu NaN olan genler için
    if is_log2:
        return X.astype(np.float32)
    return np.log2(X.clip(lower=floor, upper=ceil)).astype(np.float32)


def describe_values(X_raw):
    """Sonlu değerler üzerinden kalite özeti (NaN/INF varken min/maks 'nan' yazmasın diye)."""
    v = X_raw.values.astype(float)
    fin = v[np.isfinite(v)]
    return {
        "n_nan": int(np.isnan(v).sum()), "n_inf": int(np.isinf(v).sum()),
        "n_negative": int((fin < 0).sum()), "n_zero": int((fin == 0).sum()),
        "pct_negative_or_zero": float((fin <= 0).mean() * 100),
        "n_below_floor_20": int((fin < FLOOR).sum()), "n_above_ceil_16000": int((fin > CEIL).sum()),
        "min": float(fin.min()), "p01": float(np.percentile(fin, 1)), "median": float(np.median(fin)),
        "p99": float(np.percentile(fin, 99)), "max": float(fin.max()),
    }


def welch_de(X_log, y):
    """Welch t-testi (AML=1 vs ALL=0) + Benjamini-Hochberg FDR. BETİMSEL kullanım içindir."""
    a = X_log[y.values == 1].values.astype(float)
    b = X_log[y.values == 0].values.astype(float)
    t, p = stats.ttest_ind(a, b, equal_var=False, axis=0)
    t = np.where(np.isfinite(t), t, 0.0)
    p = np.where(np.isfinite(p), p, 1.0)
    return pd.DataFrame({"t_stat": t, "abs_t": np.abs(t), "p_value": p,
                         "fdr": false_discovery_control(p, method="bh"),
                         "log2FC_AML_vs_ALL": a.mean(axis=0) - b.mean(axis=0)}, index=X_log.columns)


def eta_squared(values, groups):
    """Bir PC skorunun bir kategorik değişkenle ilişkisi (varyansın kaç %'i gruplar arası)."""
    g = np.asarray(pd.Series(groups).astype(str))
    mu = values.mean()
    sst = float(((values - mu) ** 2).sum())
    if sst == 0:
        return float("nan")
    ssb = sum(len(values[g == k]) * (values[g == k].mean() - mu) ** 2 for k in np.unique(g))
    return float(ssb / sst)


def domain_auc(X, labels, n_pc=10):
    """İki grubun ifade profilinden ayırt edilebilirliği (5-kat CV AUC). ~0.5: fark yok; ~1.0: güçlü batch/platform
    etkisi. Denetimsiz PCA + lojistik regresyon; BETİMSEL bir teşhis ölçüsüdür."""
    Xs = StandardScaler().fit_transform(np.asarray(X, dtype=float))
    k = int(min(n_pc, Xs.shape[0] - 1, Xs.shape[1]))
    P = PCA(n_components=k, random_state=RANDOM_SEED).fit_transform(Xs)
    cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_SEED)
    proba = cross_val_predict(LogisticRegression(max_iter=2000, class_weight="balanced"),
                              P, np.asarray(labels), cv=cv, method="predict_proba")[:, 1]
    return float(roc_auc_score(labels, proba))


def stratified_sample(y, n_all=8, n_aml=7):
    """Boxplotlar için sınıfa göre dengeli rastgele örnek (v1'deki 'ilk 15 hasta' temsilî değildi)."""
    rng = np.random.default_rng(RANDOM_SEED)
    a = rng.choice(y[y == 0].index.values, min(n_all, (y == 0).sum()), replace=False)
    b = rng.choice(y[y == 1].index.values, min(n_aml, (y == 1).sum()), replace=False)
    return list(a) + list(b)


def pca_tsne_figure(X_log, y, group, name, title, top_var=None, perplexity=15):
    """PCA (scree dahil) + t-SNE şekilleri; PC skorlarını ve açıklanan varyansı döndürür."""
    X = X_log
    if top_var is not None and X.shape[1] > top_var:
        X = X[X.var(axis=0).nlargest(top_var).index]  # en yüksek varyanslı öznitelikler
    Xs = StandardScaler().fit_transform(X.values.astype(float))
    ncomp = int(min(10, Xs.shape[0] - 1, Xs.shape[1]))
    pca = PCA(n_components=ncomp, random_state=RANDOM_SEED)
    S = pca.fit_transform(Xs)
    evr = pca.explained_variance_ratio_
    v1, v2 = evr[0] * 100, evr[1] * 100
    P50 = PCA(n_components=int(min(50, Xs.shape[0] - 1, Xs.shape[1])), random_state=RANDOM_SEED).fit_transform(Xs)
    T = TSNE(n_components=2, perplexity=perplexity, random_state=RANDOM_SEED, init="pca").fit_transform(P50)

    class_names = y.map({0: "ALL", 1: "AML"}).values
    grp = np.asarray(group)
    legend_b = len(np.unique(grp)) <= 10
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    sns.scatterplot(x=S[:, 0], y=S[:, 1], hue=class_names, style=grp if legend_b else None,
                    palette=PAL_CLASS, s=45, alpha=0.85, ax=axes[0])
    axes[0].set_title(f"(a) PCA - Sınıfa Göre (PC1: %{v1:.1f}, PC2: %{v2:.1f})")
    axes[0].set_xlabel(f"PC1 (%{v1:.1f} Varyans)")
    axes[0].set_ylabel(f"PC2 (%{v2:.1f} Varyans)")
    sns.scatterplot(x=S[:, 0], y=S[:, 1], hue=grp, style=class_names, s=45, alpha=0.85,
                    palette="Dark2" if legend_b else "tab20", legend=legend_b, ax=axes[1])
    axes[1].set_title(f"(b) PCA - {title}")
    axes[1].set_xlabel(f"PC1 (%{v1:.1f} Varyans)")
    axes[1].set_ylabel(f"PC2 (%{v2:.1f} Varyans)")
    sns.scatterplot(x=T[:, 0], y=T[:, 1], hue=class_names, style=grp if legend_b else None,
                    palette=PAL_CLASS, s=45, alpha=0.85, ax=axes[2])
    axes[2].set_title(f"(c) t-SNE (Perplexity={perplexity})")
    axes[2].set_xlabel("t-SNE Boyut 1")
    axes[2].set_ylabel("t-SNE Boyut 2")
    save_fig(f"{name}_pca_tsne")

    fig, ax = plt.subplots(figsize=(5, 3.8))
    ax.bar(range(1, ncomp + 1), evr * 100, color="#4c78a8")
    ax.plot(range(1, ncomp + 1), np.cumsum(evr) * 100, "k.-")
    ax.set_xlabel("Bileşen")
    ax.set_ylabel("Açıklanan varyans (%)")
    ax.set_title("Scree + kümülatif varyans")
    save_fig(f"{name}_pca_scree")
    return S, evr


def heatmap_top30(X_log, y, de, name, title, label_func=None):
    """En ayırt edici 30 öznitelik için hiyerarşik kümelemeli ısı haritası (BETİMSEL)."""
    top = de.sort_values("abs_t", ascending=False).head(30).index
    Z = pd.DataFrame(StandardScaler().fit_transform(X_log[top].values.astype(float)),
                     index=X_log.index, columns=[label_func(p) if label_func else p for p in top]).T
    col_colors = y.map({0: PAL_CLASS["ALL"], 1: PAL_CLASS["AML"]})
    g = sns.clustermap(Z, cmap="RdBu_r", center=0, vmin=-2.5, vmax=2.5, col_colors=col_colors,
                       figsize=(13, 9), xticklabels=False, yticklabels=True)
    g.fig.suptitle(title, y=1.02)
    save_clustermap(g, name)


def load_gpl_annot(path):
    """GEO GPL anotasyon dosyasından (GPL570.annot[.gz]) prob -> gen sembolü / GenBank erişim tablosu."""
    opener = gzip.open if str(path).endswith(".gz") else open
    lines, started = [], False
    with opener(path, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            if line.startswith("!platform_table_begin"):
                started = True
                continue
            if line.startswith("!platform_table_end"):
                break
            if started:
                lines.append(line)
    df = pd.read_csv(io.StringIO("".join(lines)), sep="\t", low_memory=False, dtype=str)
    low = {c.lower(): c for c in df.columns}
    sym_col = next(c for k, c in low.items() if "gene symbol" in k or k == "gene_symbol")
    gb_col = next((c for k, c in low.items() if "genbank" in k or k == "gb_acc"), None)
    out = pd.DataFrame({"ID": df[low["id"]],
                        "symbol": df[sym_col].fillna("").str.split("///").str[0].str.strip()})
    out["gb"] = df[gb_col].fillna("").str.split("///").str[0].str.strip() if gb_col else ""
    return out[(out["symbol"] != "") & (out["symbol"] != "---")]


def golub_symbol_map(golub_ids, gpl, override_csv=None):
    """Golub (Hu6800) prob -> gen sembolü. KISMİ kapsama: GenBank erişim numarası üzerinden GPL570 ile eşleştirir.
    Kapsama oranını raporla; iyileştirmek için GOLUB_MAP_CSV ('probe,symbol') ver."""
    if override_csv:
        m = pd.read_csv(override_csv)
        m.columns = [c.strip().lower() for c in m.columns]
        return m.set_index(m.columns[0])[m.columns[1]].astype(str)
    gb2sym = gpl[gpl["gb"] != ""].drop_duplicates("gb").set_index("gb")["symbol"]
    res = {}
    for gid in golub_ids:
        base = str(gid).split("#")[0]
        if base.startswith(("AFFX", "HG")):
            continue
        acc = base.split("_")[0]
        if acc in gb2sym.index:
            res[gid] = gb2sym[acc]
    return pd.Series(res, dtype=str)


def collapse_to_symbols(X_log, f2s):
    """Aynı gen sembolüne giden öznitelikler arasında ortalaması en yüksek olanı tutar."""
    s = f2s[f2s.index.isin(X_log.columns)].dropna()
    s = s[s.astype(str) != ""]
    if s.empty:
        return pd.DataFrame(index=X_log.index)
    d = pd.DataFrame({"sym": s.values, "mean": X_log[s.index].mean(axis=0).values}, index=s.index)
    d = d.sort_values("mean", ascending=False).drop_duplicates("sym")
    G = X_log[d.index].copy()
    G.columns = d["sym"].values
    return G


In [27]:
# ==============================================================================
# ADIM 1: GOLUB (1999) VERİ SETİNİN AÇILMASI, DOĞRULANMASI VE TEMİZLENMESİ
# ==============================================================================
def load_and_verify_golub(zip_path=GOLUB_ZIP_PATH, extract_dir=EXTRACT_DIR):
    """
    Golub (1999) ALL/AML veri setini zipten çıkarır, 'call' (P/A/M) sütunlarını
    analiz edip kaldırır ve hasta numaralarını actual.csv ile birebir eşleştirir.
    """
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(extract_dir)
        print(f"[BİLGİ] Golub zip dosyası '{extract_dir}' klasörüne çıkarıldı.")

    train_df = pd.read_csv(os.path.join(extract_dir, "data_set_ALL_AML_train.csv"))
    test_df = pd.read_csv(os.path.join(extract_dir, "data_set_ALL_AML_independent.csv"))
    actual_df = pd.read_csv(os.path.join(extract_dir, "actual.csv"))
    R = RESULTS["golub"]

    # 1.0. Sütun sırası kontrolü
    train_sample_cols = [c for c in train_df.columns if re.fullmatch(r"\d+", str(c).strip())]
    test_sample_cols = [c for c in test_df.columns if re.fullmatch(r"\d+", str(c).strip())]
    tr_order, te_order = [int(c) for c in train_sample_cols], [int(c) for c in test_sample_cols]
    R["train_column_order_is_numeric"] = tr_order == sorted(tr_order)
    R["test_column_order_is_numeric"] = te_order == sorted(te_order)
    print(f"[KONTROL] Eğitim sütun sırası numaralı mı: {R['train_column_order_is_numeric']} | "
          f"Test: {R['test_column_order_is_numeric']}")
    print(f"[KONTROL] Eğitim ilk 12 sütun: {tr_order[:12]} | Test ilk 12 sütun: {te_order[:12]}")

    # Eğitim ve test aynı gen satırlarına mı sahip?
    acc_tr = train_df["Gene Accession Number"].astype(str).str.strip()
    acc_te = test_df["Gene Accession Number"].astype(str).str.strip()
    if len(acc_tr) != len(acc_te) or not (acc_tr.values == acc_te.values).all():
        raise ValueError("Eğitim ve test dosyalarının gen satırları aynı değil; eşleştirme gerekir.")
    dup = acc_tr.groupby(acc_tr).cumcount()
    acc_u = pd.Series(np.where(dup > 0, acc_tr + "#" + dup.astype(str), acc_tr), index=acc_tr.index)
    R["n_duplicate_accessions"] = int((dup > 0).sum())

    # 1.1. Call (P/A/M) sütunlarının istatistiği
    train_call_cols = [c for c in train_df.columns if "call" in str(c).lower()]
    test_call_cols = [c for c in test_df.columns if "call" in str(c).lower()]
    all_calls = pd.concat([train_df[train_call_cols], test_df[test_call_cols]], axis=1)
    all_calls = all_calls.astype(str).apply(lambda col: col.str.strip().str.upper())
    call_counts = pd.Series(all_calls.values.ravel()).value_counts(normalize=True) * 100
    frac_absent = (all_calls == "A").mean(axis=1)
    R["call_percent"] = {k: float(v) for k, v in call_counts.items()}
    R["genes_absent_in_ge90pct_samples"] = int((frac_absent >= 0.9).sum())
    print("\n--- 1.1. GOLUB CALL (P/A/M) SÜTUN DAĞILIMI (%) ---")
    print(call_counts.round(2))
    print(f"Örneklerin >= %90'ında 'Absent' çağrılan gen sayısı: {R['genes_absent_in_ge90pct_samples']}")

    # 1.2. Sayısal olmayan 'call' sütunlarının kaldırılması ve hasta sütunlarının ayrılması
    gene_meta = pd.DataFrame({"Gene Description": train_df["Gene Description"].values,
                              "Gene Accession Number": acc_u.values})
    X_train_raw = train_df[train_sample_cols].apply(pd.to_numeric, errors="coerce").T
    X_test_raw = test_df[test_sample_cols].apply(pd.to_numeric, errors="coerce").T
    X_train_raw.index = X_train_raw.index.astype(int)
    X_test_raw.index = X_test_raw.index.astype(int)

    # 1.3. Hasta numarasına göre sıralama (1..38 eğitim, 39..72 bağımsız test)
    X_train_raw = X_train_raw.sort_index()
    X_test_raw = X_test_raw.sort_index()
    X_train_raw.columns = gene_meta["Gene Accession Number"].values
    X_test_raw.columns = gene_meta["Gene Accession Number"].values

    # 1.4. actual.csv ile hasta ID üzerinden kesin eşleştirme ve doğrulama
    actual_map = actual_df.set_index("patient")["cancer"].astype(str).str.strip().str.upper().to_dict()
    y_train_str = pd.Series([actual_map[i] for i in X_train_raw.index], index=X_train_raw.index)
    y_test_str = pd.Series([actual_map[i] for i in X_test_raw.index], index=X_test_raw.index)

    # Etiket kodlaması: ALL = 0, AML = 1 (Biyolojik olarak ikisi de lösemi alt tipidir)
    y_train = (y_train_str == "AML").astype(int)
    y_test = (y_test_str == "AML").astype(int)

    R["counts"] = {"train": {"ALL": int((y_train == 0).sum()), "AML": int((y_train == 1).sum())},
                   "test": {"ALL": int((y_test == 0).sum()), "AML": int((y_test == 1).sum())}}
    R["counts_match_guide"] = R["counts"] == {"train": {"ALL": 27, "AML": 11}, "test": {"ALL": 20, "AML": 14}}
    print("\n--- 1.2. GOLUB DOĞRULANMIŞ ÖRNEK VE SINIF DAĞILIMI ---")
    print(f"Eğitim Seti (n={len(X_train_raw)}): ALL={(y_train==0).sum()}, AML={(y_train==1).sum()}")
    print(f"Bağımsız Test (n={len(X_test_raw)}): ALL={(y_test==0).sum()}, AML={(y_test==1).sum()}")
    print(f"Rehberdeki sayılarla uyumlu mu: {R['counts_match_guide']}")

    # 1.5. Veri kalitesi kontrolü: NaN, INF, negatif, eşik dışı ve SONLU min/maks
    X_all_raw = pd.concat([X_train_raw, X_test_raw], axis=0)
    R["quality"] = describe_values(X_all_raw)
    R["n_samples"], R["n_features"] = int(X_all_raw.shape[0]), int(X_all_raw.shape[1])
    print("\n--- 1.3. GOLUB VERİ KALİTESİ ÖZETİ ---")
    for k, v in R["quality"].items():
        print(f"{k}: {v}")
    pd.Series(R["quality"]).rename("value").to_frame().to_csv(os.path.join(TABLE_DIR, "golub_quality.csv"))

    return X_train_raw, X_test_raw, y_train, y_test, gene_meta


In [28]:
X_tr, X_te, y_tr, y_te, gene_meta = load_and_verify_golub()

[BİLGİ] Golub zip dosyası './data_golub' klasörüne çıkarıldı.
[KONTROL] Eğitim sütun sırası numaralı mı: False | Test: False
[KONTROL] Eğitim ilk 12 sütun: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] | Test ilk 12 sütun: [39, 40, 42, 47, 48, 49, 41, 43, 44, 45, 46, 70]

--- 1.1. GOLUB CALL (P/A/M) SÜTUN DAĞILIMI (%) ---
A    70.16
P    28.23
M     1.61
Name: proportion, dtype: float64
Örneklerin >= %90'ında 'Absent' çağrılan gen sayısı: 3416

--- 1.2. GOLUB DOĞRULANMIŞ ÖRNEK VE SINIF DAĞILIMI ---
Eğitim Seti (n=38): ALL=27, AML=11
Bağımsız Test (n=34): ALL=20, AML=14
Rehberdeki sayılarla uyumlu mu: True

--- 1.3. GOLUB VERİ KALİTESİ ÖZETİ ---
n_nan: 0
n_inf: 0
n_negative: 147174
n_zero: 1615
pct_negative_or_zero: 28.987430058758434
n_below_floor_20: 167957
n_above_ceil_16000: 4259
min: -28400.0
p01: -982.0
median: 120.0
p99: 14750.390000000014
max: 71369.0


In [29]:
# ==============================================================================
# ADIM 2: GOLUB ÖN İŞLEME (KIRPMA + LOG2 DÖNÜŞÜMÜ) VE EDA GÖRSELLERİ
# ==============================================================================
def preprocess_and_plot_golub(X_train_raw, X_test_raw, y_train, y_test, gene_meta):

    R = RESULTS["golub"]
    X_all_raw = pd.concat([X_train_raw, X_test_raw], axis=0)
    y_all = pd.concat([y_train, y_test], axis=0)
    n_tr, n_te = len(X_train_raw), len(X_test_raw)
    split_labels = pd.Series([f"Eğitim (n={n_tr})"] * n_tr + [f"Bağımsız Test (n={n_te})"] * n_te,
                             index=X_all_raw.index)

    # INF/NaN varsa eğitim ortalaması ile doldurma + [20, 16000] eşik kırpma + log2 (yalnızca BETİMSEL EDA için)
    X_log2 = clean_for_eda(X_all_raw, train_idx=X_train_raw.index)
    raw_fin = X_all_raw.values[np.isfinite(X_all_raw.values)]
    R["skew_raw"], R["skew_log2"] = float(stats.skew(raw_fin)), float(stats.skew(X_log2.values.ravel()))
    R["log2_mean_sd"] = [float(X_log2.values.mean()), float(X_log2.values.std())]

    # ŞEKİL 1: Sınıf Dağılımı ve Log2 Dönüşümü Öncesi / Sonrası İfade Dağılımı
    idx = stratified_sample(y_all)
    raw_plot = X_all_raw.loc[idx].replace([np.inf, -np.inf], np.nan)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    sns.countplot(x=y_all.map({0: "ALL (0)", 1: "AML (1)"}), hue=split_labels, ax=axes[0], palette="Set2")
    axes[0].set_title("(a) Golub Sınıf ve Bölünme Dağılımı")
    axes[0].set_xlabel("Lösemi Alt Tipi")
    axes[0].set_ylabel("Hasta Sayısı (n)")
    sns.boxplot(data=raw_plot.T, ax=axes[1], color="salmon", fliersize=1)
    axes[1].set_title(f"(b) Log2 Öncesi Ham İfade (Sınıfa Göre Rastgele {len(idx)} Hasta)")
    axes[1].set_xlabel("Hasta Numarası")
    axes[1].set_ylabel("Ham Floresan Yoğunluğu")
    sns.boxplot(data=X_log2.loc[idx].T, ax=axes[2], color="skyblue", fliersize=1)
    axes[2].set_title("(c) Kırpma [20-16k] + Log2 Sonrası Dağılım")
    axes[2].set_xlabel("Hasta Numarası")
    axes[2].set_ylabel("Log2(İfade Düzeyi)")
    save_fig("fig1_golub_distributions")

    # ŞEKİL 2: Boyut İndirgeme (PCA ve t-SNE) + Eğitim/Test Batch Etkisi (+ scree)
    S, evr = pca_tsne_figure(X_log2, y_all, split_labels.values, "fig2_golub", "Eğitim vs. Test (Batch Etkisi Kontrolü)")
    R["pca"] = {"explained_variance_ratio": [float(v) for v in evr], "cum_pc1_2": float(evr[:2].sum()),
                "cum_pc1_10": float(evr.sum())}
    R["pc_association_eta2"] = [{"PC": k + 1, "sinif": eta_squared(S[:, k], y_all.values),
                                 "split": eta_squared(S[:, k], split_labels.values)} for k in range(5)]
    R["batch_auc_train_vs_test"] = domain_auc(X_log2.values, (split_labels.str.startswith("Bağımsız")).astype(int).values)
    print(f"\n[BATCH] Eğitim/test ayrımının ifadeden öngörülebilirliği (AUC): {R['batch_auc_train_vs_test']:.3f}")

    # ŞEKİL 3: Sadece Eğitim Setinde Welch t-testi (FDR'li) ile En Ayırt Edici 30 Gen ve Isı Haritası
    X_tr_log = X_log2.loc[X_train_raw.index]
    de = welch_de(X_tr_log, y_train)
    gene_stats = de.copy()
    gene_stats.insert(0, "Gene_Description", gene_meta["Gene Description"].values)
    gene_stats.insert(0, "Probe_ID", X_tr_log.columns)
    gene_stats = gene_stats.sort_values("abs_t", ascending=False)
    gene_stats.head(100).to_csv(os.path.join(TABLE_DIR, "golub_train_de_top100.csv"), index=False)
    R["de_train"] = {"n_fdr_lt_0.05": int((de["fdr"] < 0.05).sum()),
                     "n_fdr_lt_0.05_and_abs_log2FC_gt_1": int(((de["fdr"] < 0.05) & (de["log2FC_AML_vs_ALL"].abs() > 1)).sum()),
                     "top10_probes": list(gene_stats.head(10)["Probe_ID"])}
    print("\n--- 2.1. GOLUB EĞİTİM SETİNDE EN AYIRT EDİCİ İLK 15 GEN (Sızıntısız seçim) ---")
    print(gene_stats.head(15)[["Probe_ID", "Gene_Description", "t_stat", "p_value", "fdr"]].to_string(index=False))
    print(f"FDR < 0.05 olan öznitelik sayısı: {R['de_train']['n_fdr_lt_0.05']}")

    desc_map = dict(zip(gene_meta["Gene Accession Number"], gene_meta["Gene Description"]))
    heatmap_top30(X_log2, y_all, de, "fig3_golub_top30_heatmap",
                  "Golub En Ayırt Edici 30 Gen Hiyerarşik Kümeleme Isı Haritası (Mavi: ALL, Kırmızı: AML)",
                  label_func=lambda p: f"{p} ({str(desc_map.get(p, ''))[:22]})")

    # Batch: aynı sınıf içinde eğitim vs. test farklı gen sayısı (FDR < 0.05)
    within = {}
    for k, name in ((0, "ALL"), (1, "AML")):
        tr_idx = [i for i in X_train_raw.index if y_train.loc[i] == k]
        te_idx = [i for i in X_test_raw.index if y_test.loc[i] == k]
        t, p = stats.ttest_ind(X_log2.loc[te_idx].values.astype(float), X_log2.loc[tr_idx].values.astype(float),
                               equal_var=False, axis=0)
        p = np.where(np.isfinite(p), p, 1.0)
        within[name] = int((false_discovery_control(p, method="bh") < 0.05).sum())
    R["batch_genes_fdr_lt_0.05_test_vs_train_within_class"] = within

    return X_log2, y_all, gene_stats


In [30]:
X_golub_log2, y_golub, golub_stats = preprocess_and_plot_golub(X_tr, X_te, y_tr, y_te, gene_meta)


[BATCH] Eğitim/test ayrımının ifadeden öngörülebilirliği (AUC): 0.772


/usr/local/lib/python3.13/dist-packages/scipy/stats/_axis_nan_policy.py:611: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, axis=axis, **kwds)



--- 2.1. GOLUB EĞİTİM SETİNDE EN AYIRT EDİCİ İLK 15 GEN (Sızıntısız seçim) ---
        Probe_ID                                                                                                Gene_Description    t_stat      p_value      fdr
       M27891_at                                                    CST3 Cystatin C (amyloid angiopathy and cerebral hemorrhage)  9.532188 4.392537e-10 0.000003
  U50136_rna1_at                                                                            Leukotriene C4 synthase (LTC4S) gene  8.480939 4.972650e-08 0.000040
       X95735_at                                                                                                           Zyxin  8.114032 1.222489e-09 0.000004
       X52142_at                                                                                             CTPS CTP synthetase -8.050984 1.569947e-08 0.000022
U22376_cds2_s_at C-myb gene extracted from Human (c-myb) gene; complete primary cds; and five complete alternativel

/usr/local/lib/python3.13/dist-packages/scipy/stats/_axis_nan_policy.py:611: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, axis=axis, **kwds)
/usr/local/lib/python3.13/dist-packages/scipy/stats/_axis_nan_policy.py:611: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, axis=axis, **kwds)


In [31]:
# ==============================================================================
# ADIM 3: GSE13159 (MILE) VERİSİNİN OKUNMASI, ALL/AML FİLTRELENMESİ VE EDA
# ==============================================================================
def map_main_lineage(text):
    """ALL/AML ana soyunu belirler (CLL, CML, MDS ve sağlıklı kontroller dışlanır).
    Sözcük sınırlı regex kullanır (alt dize eşleşmesi 'SMALL' gibi sözcüklerde yanlış eşleşirdi)."""
    t = str(text)
    if re.search(r"(?<![A-Za-z])(CLL|CML|MDS)(?![A-Za-z])|non-leukemia|healthy", t, re.I):
        return "EXCLUDE"
    has_all = re.search(r"(?<![A-Za-z])ALL(?![A-Za-z])", t) is not None
    has_aml = re.search(r"(?<![A-Za-z])AML(?![A-Za-z])", t) is not None
    if has_all and has_aml:
        return "AMBIGUOUS"
    if has_all:
        return "ALL"
    if has_aml:
        return "AML"
    return "EXCLUDE"


def load_and_analyze_gse13159(gz_path=GSE13159_GZ_PATH):
    """
    GSE13159_series_matrix.txt.gz dosyasından hasta alt tip meta verisini ve
    gen ifade matrisini okur; sadece ALL ve AML örneklerini filtreler. Örnek sayısını KODUN
    yazdırdığı değerden raporla (sabit sayı yazma).
    """
    if not os.path.exists(gz_path):
        print(f"[UYARI] {gz_path} bulunamadı. Lütfen Drive dosya yolunu kontrol edin.")
        return None, None, None
    R = RESULTS["gse13159"]

    sample_ids, char_rows, id_col = [], [], "ID_REF"
    with gzip.open(gz_path, "rt", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if line.startswith("!Sample_geo_accession"):
                sample_ids = [x.strip('"') for x in line.strip().split("\t")[1:]]
            elif line.startswith("!Sample_characteristics_ch1"):
                char_rows.append([x.strip('"') for x in line.rstrip("\n").split("\t")[1:]])
            elif line.startswith("!series_matrix_table_begin"):
                id_col = next(f).rstrip("\n").split("\t")[0].strip('"')
                break

    cls_row = next((r for r in char_rows if any("leukemia class" in c.lower() for c in r)), None)
    if cls_row is None:
        prefixes = sorted({c.split(":")[0] for r in char_rows for c in r[:3]})
        raise ValueError(f"'leukemia class' satırı bulunamadı. Mevcut karakteristik anahtarları: {prefixes}")
    center_row = next((r for r in char_rows if any(re.match(r"\s*(center|centre|laborator|site)", c, re.I) for c in r[:5])), None)

    # İfade tablosunu okuma (satırlar: prob, sütunlar: örnek); float32 ile bellek tasarrufu
    dt = defaultdict(lambda: np.float32)
    dt[id_col] = str
    expr_df = pd.read_csv(gz_path, sep="\t", comment="!", dtype=dt, na_values=["null", "NA"]).set_index(id_col)

    meta_df = pd.DataFrame({"Sample_ID": sample_ids, "Raw_Class": cls_row}).set_index("Sample_ID")
    if center_row is not None:
        meta_df["Center"] = [c.split(":", 1)[-1].strip() for c in center_row]
    meta_df["Lineage"] = meta_df["Raw_Class"].apply(map_main_lineage)
    meta_df["Raw_Class"] = meta_df["Raw_Class"].str.replace("leukemia class: ", "", case=False)

    # Sınıf eşlemesini kaydet: ELLE doğrula (JCO 2010 Tablo 1 ile karşılaştır)
    pd.crosstab(meta_df["Raw_Class"], meta_df["Lineage"]).to_csv(os.path.join(TABLE_DIR, "mile_class_mapping_check.csv"))
    R["n_series_samples"] = int(expr_df.shape[1])
    R["n_probes"] = int(expr_df.shape[0])
    R["lineage_counts_all_samples"] = {k: int(v) for k, v in meta_df["Lineage"].value_counts().items()}

    valid_meta = meta_df[meta_df["Lineage"].isin(["ALL", "AML"])].copy()
    X_mile_raw = expr_df[valid_meta.index].T
    del expr_df
    y_mile = (valid_meta["Lineage"] == "AML").astype(int)
    R["n_samples"] = int(len(X_mile_raw))
    R["counts"] = {"ALL": int((y_mile == 0).sum()), "AML": int((y_mile == 1).sum())}

    print("\n--- 3.1. GSE13159 (MILE) DOĞRULANMIŞ ÖRNEK VE SINIF DAĞILIMI ---")
    print(f"Toplam Kohort: {R['n_series_samples']} örnek | Prob: {R['n_probes']} | Filtrelenen Akut Lösemi: {len(X_mile_raw)} örnek")
    print(valid_meta["Lineage"].value_counts())
    print("\nAlt Tip Dağılımı:")
    print(valid_meta["Raw_Class"].value_counts())
    print("(Sınıf eşlemesini tables/mile_class_mapping_check.csv ile JCO 2010 Tablo 1'e karşı doğrulayın.)")

    # Değer ölçeği kontrolü: dosyadaki değerler zaten log2 mi?
    mx, mn = float(np.nanmax(X_mile_raw.values)), float(np.nanmin(X_mile_raw.values))
    is_log2 = mx < 50
    R["quality"] = describe_values(X_mile_raw)
    R["value_range"] = [mn, mx]
    R["values_look_log2"] = bool(is_log2)
    print(f"\n[KONTROL] MILE değer aralığı: [{mn:.2f}, {mx:.2f}] -> {'log2 ölçeği gibi görünüyor' if is_log2 else 'ham yoğunluk gibi görünüyor, log2 uygulanacak'}")
    pd.Series(R["quality"]).rename("value").to_frame().to_csv(os.path.join(TABLE_DIR, "gse13159_quality.csv"))
    X_mile = clean_for_eda(X_mile_raw, is_log2=is_log2, floor=1.0, ceil=np.inf)

    # ŞEKİL 4: Sınıf/alt tip dağılımı ve değer dağılımı (önce/sonra)
    idx = stratified_sample(y_mile)
    raw_plot = X_mile_raw.loc[idx].replace([np.inf, -np.inf], np.nan)
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    subtype_counts = valid_meta["Raw_Class"].value_counts()
    sns.barplot(x=subtype_counts.values, y=subtype_counts.index, hue=subtype_counts.index, palette="viridis",
                legend=False, ax=axes[0])
    axes[0].set_title("(a) GSE13159 Moleküler Alt Tip Dağılımı")
    axes[0].set_xlabel("Hasta Sayısı (n)")
    axes[0].set_ylabel("Alt Tip")
    sns.boxplot(data=raw_plot.T, ax=axes[1], color="salmon", fliersize=1)
    axes[1].set_title("(b) Dosyadaki Değerler (Rastgele 15 Hasta)" + (" [zaten log2]" if is_log2 else ""))
    axes[1].set_xlabel("Örnek")
    axes[1].set_ylabel("Değer")
    axes[1].set_xticks([])
    sns.boxplot(data=X_mile.loc[idx].T, ax=axes[2], color="skyblue", fliersize=1)
    axes[2].set_title("(c) Log2 Ölçeğinde Dağılım")
    axes[2].set_xlabel("Örnek")
    axes[2].set_ylabel("Log2(İfade Düzeyi)")
    axes[2].set_xticks([])
    save_fig("fig4_gse13159_subtypes_distributions")

    # ŞEKİL 5: PCA (en yüksek varyanslı 5.000 prob) + t-SNE; grup: merkez varsa merkez, yoksa alt tip
    group = valid_meta["Center"].values if "Center" in valid_meta else valid_meta["Raw_Class"].values
    title = "Merkez" if "Center" in valid_meta else "Alt Tip"
    S, evr = pca_tsne_figure(X_mile, y_mile, group, "fig5_gse13159", title, top_var=5000, perplexity=30)
    R["pca"] = {"n_features_used": 5000, "explained_variance_ratio": [float(v) for v in evr],
                "cum_pc1_2": float(evr[:2].sum()), "cum_pc1_10": float(evr.sum())}
    assoc = []
    for k in range(5):
        row = {"PC": k + 1, "sinif": eta_squared(S[:, k], y_mile.values),
               "alt_tip": eta_squared(S[:, k], valid_meta["Raw_Class"].values)}
        if "Center" in valid_meta:
            row["merkez"] = eta_squared(S[:, k], valid_meta["Center"].values)
        assoc.append(row)
    R["pc_association_eta2"] = assoc
    pd.DataFrame(assoc).to_csv(os.path.join(TABLE_DIR, "gse13159_pc_association.csv"), index=False)

    # ŞEKİL 6: Welch t-testi (FDR'li) ve en ayırt edici 30 prob ısı haritası (BETİMSEL)
    de = welch_de(X_mile, y_mile)
    de.sort_values("abs_t", ascending=False).head(100).to_csv(os.path.join(TABLE_DIR, "gse13159_de_top100.csv"))
    R["de"] = {"n_features": int(de.shape[0]), "n_fdr_lt_0.05": int((de["fdr"] < 0.05).sum()),
               "n_fdr_lt_0.05_and_abs_log2FC_gt_1": int(((de["fdr"] < 0.05) & (de["log2FC_AML_vs_ALL"].abs() > 1)).sum()),
               "top10_probes": list(de.sort_values("abs_t", ascending=False).head(10).index)}
    print(f"\nGSE13159 FDR < 0.05 olan prob sayısı: {R['de']['n_fdr_lt_0.05']} / {R['de']['n_features']}")
    heatmap_top30(X_mile, y_mile, de, "fig6_gse13159_top30_heatmap",
                  "GSE13159 En Ayırt Edici 30 Prob Hiyerarşik Kümeleme Isı Haritası (Mavi: ALL, Kırmızı: AML)")

    return X_mile, y_mile, valid_meta


In [32]:
X_mile, y_mile, mile_meta = load_and_analyze_gse13159()


--- 3.1. GSE13159 (MILE) DOĞRULANMIŞ ÖRNEK VE SINIF DAĞILIMI ---
Toplam Kohort: 2096 örnek | Prob: 54675 | Filtrelenen Akut Lösemi: 1292 örnek
Lineage
ALL    750
AML    542
Name: count, dtype: int64

Alt Tip Dağılımı:
Raw_Class
AML with normal karyotype + other abnormalities    351
c-ALL/Pre-B-ALL without t(9;22)                    237
T-ALL                                              174
c-ALL/Pre-B-ALL with t(9;22)                       122
Pro-B-ALL with t(11q23)/MLL                         70
ALL with t(12;21)                                   58
AML complex aberrant karyotype                      48
ALL with hyperdiploid karyotype                     40
AML with t(8;21)                                    40
AML with t(11q23)/MLL                               38
AML with t(15;17)                                   37
ALL with t(1;19)                                    36
AML with inv(16)/t(16;16)                           28
mature B-ALL with t(8;14)                           13
N

/usr/local/lib/python3.13/dist-packages/seaborn/matrix.py:560: UserWarning: Clustering large matrix with scipy. Installing `fastcluster` may give better performance.
  warnings.warn(msg)
/usr/local/lib/python3.13/dist-packages/seaborn/matrix.py:560: UserWarning: Clustering large matrix with scipy. Installing `fastcluster` may give better performance.
  warnings.warn(msg)
